# Lab Work - 11.6

## Q1 — Similarity Score & Split Gain

**Dataset**  
$x = [1, 2, 3, 4, 5]$  
$y = [2.5, 3.5, 3.0, 5.5, 6.0]$ (regression)  
$\lambda = 1$ (L2 regularisation term)

### 01–02. Gradients and Hessians (MSE loss)

For MSE loss we use:
- $g_i = \hat{y}_i - y_i$
- $h_i = 1$ for all points

Initial prediction $F_0 = \text{mean}(y) = 4.1$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1, 2, 3, 4, 5])
y = np.array([2.5, 3.5, 3.0, 5.5, 6.0])
lam = 1.0

F0 = np.mean(y)
print(f"F0 (mean of y) = {F0}")

# Gradients and Hessians
g = F0 - y          # gi = ŷi − yi
h = np.ones_like(y) # hi = 1

print("\nGradients gi:", g)
print("Hessians  hi:", h)
print("Σg =", g.sum())
print("Σh =", h.sum())


### 03. Similarity Score for Root Node

$$\text{Similarity} = \frac{(\sum g_i)^2}{\sum h_i + \lambda}$$

In [ ]:
def similarity(g, h, lam=1.0):
    return (g.sum()**2) / (h.sum() + lam)

sim_root = similarity(g, h, lam)
print(f"Similarity Score (Root) = {sim_root:.6f}")


### 04–05. Split at $x \le 2$

- Left: $x \in \{1,2\}$
- Right: $x \in \{3,4,5\}$

In [ ]:
# Split x <= 2
mask_L = x <= 2
mask_R = ~mask_L

sim_L_2 = similarity(g[mask_L], h[mask_L], lam)
sim_R_2 = similarity(g[mask_R], h[mask_R], lam)
gain_2  = sim_L_2 + sim_R_2 - sim_root

print(f"Sim_Left  (x≤2) = {sim_L_2:.6f}")
print(f"Sim_Right (x≤2) = {sim_R_2:.6f}")
print(f"Gain (x≤2)      = {gain_2:.6f}")


### 06. Split at $x \le 3$ and comparison

In [ ]:
# Split x <= 3
mask_L = x <= 3
mask_R = ~mask_L

sim_L_3 = similarity(g[mask_L], h[mask_L], lam)
sim_R_3 = similarity(g[mask_R], h[mask_R], lam)
gain_3  = sim_L_3 + sim_R_3 - sim_root

print(f"Sim_Left  (x≤3) = {sim_L_3:.6f}")
print(f"Sim_Right (x≤3) = {sim_R_3:.6f}")
print(f"Gain (x≤3)      = {gain_3:.6f}")

print("\n--- Comparison ---")
print(f"Gain(x≤2) = {gain_2:.6f}")
print(f"Gain(x≤3) = {gain_3:.6f}")
best_split = "x ≤ 3" if gain_3 > gain_2 else "x ≤ 2"
print(f"Higher Gain → best split is **{best_split}**")


## Q2 — Leaf Weights, Pruning & Prediction

### 01. Leaf Output Weights (best split from Q1)

$$\text{Output}_{\text{leaf}} = -\frac{\sum g_i}{\sum h_i + \lambda}$$

In [ ]:
# Using best split x <= 3
mask_L = x <= 3
mask_R = ~mask_L

out_L = -g[mask_L].sum() / (h[mask_L].sum() + lam)
out_R = -g[mask_R].sum() / (h[mask_R].sum() + lam)

print(f"Output_Left  (x≤3) = {out_L:.6f}")
print(f"Output_Right (x≤3) = {out_R:.6f}")


### 02. First Tree Prediction

$$F_1(x) = F_0 + \eta \times \text{Output}_{\text{leaf}}(x), \quad \eta = 0.3$$

In [ ]:
eta = 0.3
F1 = np.where(x <= 3, F0 + eta * out_L, F0 + eta * out_R)

print("x values :", x)
print("F1(x)    :", np.round(F1, 6))


### 03. Pruning with $\gamma = 2.0$

If Gain < γ the split is pruned (node stays a leaf).

In [ ]:
gamma = 2.0
print(f"Gain of best split = {gain_3:.6f}")
print(f"γ (min gain)       = {gamma}")
if gain_3 > gamma:
    print("Gain > γ → keep the split (do NOT prune)")
else:
    print("Gain ≤ γ → prune the split, keep as single leaf")


### 04. New Residuals (become gradients for Round 2)

$$r_i = y_i - F_1(x_i)$$

In [ ]:
residuals = y - F1
print("New residuals ri:", np.round(residuals, 6))
print("(These become the new gradients gi for the next boosting round)")


### 05. Classification Variant

Targets changed to $y = [0, 0, 0, 1, 1]$

$$F_0 = \log\frac{p}{1-p}, \quad p = \text{mean}(y)$$

$$g_i = p_i - y_i, \quad h_i = p_i(1-p_i)$$
(where $p_i = \sigma(F_0)$ is the same for all instances at Round 1)

In [ ]:
y_cls = np.array([0., 0., 0., 1., 1.])
p = y_cls.mean()
F0_cls = np.log(p / (1 - p))
print(f"p = mean(y) = {p}")
print(f"F0 = logit(p) = {F0_cls:.6f}")

# At Round 1, predicted probability is the same for every sample
p_i = 1 / (1 + np.exp(-F0_cls))   # = p
print(f"p_i (sigmoid(F0)) = {p_i:.6f}")

g_cls = p_i - y_cls
h_cls = p_i * (1 - p_i)

print("\nGradients gi:", g_cls)
print("Hessians  hi:", h_cls)


### 06. Reflection — Why Hessian helps

Standard GBM uses only the first derivative (gradient). XGBoost uses both the gradient **and** the second derivative (Hessian).

- The similarity score / leaf weight formula is essentially a **Newton–Raphson** step:  
  $w^* = -\frac{G}{H+\lambda}$
- Using the curvature (Hessian) gives a better local quadratic approximation of the loss, leading to more accurate step sizes and better split decisions, especially when the loss is far from quadratic or when instance weights differ.
- Regularisation on the Hessian term also provides a natural way to control leaf complexity.

## Q3 — Visualize It

### 02. Similarity Score Bar Chart


In [ ]:
labels = ['Root', 'Left Leaf\n(x≤3)', 'Right Leaf\n(x>3)']
sims   = [sim_root, sim_L_3, sim_R_3]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, sims, color=['#4C72B0', '#55A868', '#C44E52'], edgecolor='black')
ax.set_ylabel('Similarity Score')
ax.set_title('XGBoost Similarity Scores (λ = 1)')
ax.axhline(0, color='grey', linewidth=0.8)

# Annotate Gain
ax.text(1.5, max(sims)*0.85,
        f'Gain = Sim_L + Sim_R − Sim_Root\n= {gain_3:.4f}',
        ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

for bar, val in zip(bars, sims):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


### 03. Split Diagram (textual representation of the tree)

```
                    [Root]
                 Sim = 0.0000
              (all 5 points)
                      |
              split: x ≤ 3 ?
             /              \
            /                \
   [Left Leaf]            [Right Leaf]
   x ≤ 3                  x > 3
   Sim = 2.7225           Sim = 3.6300
   Output = -0.8250       Output = 1.1000
   (3 samples)            (2 samples)
```


### 04. Prediction Curve


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Original data
ax.scatter(x, y, s=80, c='black', zorder=5, label='Original y')

# F0 flat line
ax.axhline(F0, color='steelblue', linestyle='--', linewidth=2, label=f'F0 = {F0:.1f} (mean)')

# F1 step function
xx = np.linspace(0.5, 5.5, 500)
F1_curve = np.where(xx <= 3, F0 + eta*out_L, F0 + eta*out_R)
ax.plot(xx, F1_curve, color='crimson', linewidth=2.5, label='F1 (after one tree)')

ax.set_xlabel('x')
ax.set_ylabel('y / prediction')
ax.set_title('Prediction Curve: Original y, F0 and F1')
ax.legend()
ax.set_xticks(x)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 05. Bonus — Effect of increasing λ on Leaf Output Weights


In [ ]:
lambdas = [0, 1, 5]
outputs_L = []
outputs_R = []

for l in lambdas:
    oL = -g[mask_L].sum() / (h[mask_L].sum() + l)
    oR = -g[mask_R].sum() / (h[mask_R].sum() + l)
    outputs_L.append(oL)
    outputs_R.append(oR)
    print(f"λ = {l:2d}  →  Output_Left = {oL:8.4f}   Output_Right = {oR:8.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
width = 0.35
xpos = np.arange(len(lambdas))
ax.bar(xpos - width/2, outputs_L, width, label='Left leaf (x≤3)', color='#55A868')
ax.bar(xpos + width/2, outputs_R, width, label='Right leaf (x>3)', color='#C44E52')
ax.set_xticks(xpos)
ax.set_xticklabels([f'λ={l}' for l in lambdas])
ax.set_ylabel('Leaf Output Weight')
ax.set_title('Effect of Regularisation λ on Leaf Weights\n(larger λ shrinks the weights toward 0)')
ax.axhline(0, color='grey', linewidth=0.8)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## Q4 — Explain It Cold

### 01. What is the XGBoost Similarity Score?

The **Similarity Score** of a node is

$$S = \frac{(\sum_{i\in\text{node}} g_i)^2}{\sum_{i\in\text{node}} h_i + \lambda}$$

It measures how pure / how much the node reduces the loss (under a second-order approximation).  
A higher score means the gradients in that node are more consistent in sign and magnitude, so a single leaf weight can reduce the loss a lot.

- **Numerator** – squared sum of gradients (how strong the collective residual is).  
- **Denominator** – sum of Hessians (local curvature / effective sample weight) plus the L2 regularisation term λ.  
- **Role of λ** – penalises leaves that contain few samples or small Hessians, preventing the model from creating overly confident leaves on tiny groups of points. It shrinks the effective score of small/noisy leaves.

### 02. What is the Gain in XGBoost?

$$\text{Gain} = S_{\text{Left}} + S_{\text{Right}} - S_{\text{Parent}}$$

(sometimes written with an extra $-\gamma$ term).

Gain quantifies the **reduction in the loss approximation** obtained by performing the split.  
XGBoost enumerates candidate split points and keeps the one with the highest Gain.  
If the best Gain is still smaller than the threshold γ, the split is discarded (pre-pruning).

### 03. What is the γ (gamma) parameter?

γ is the **minimum loss reduction** required to make a further split on a leaf node.  
It is a pre-pruning (regularisation) parameter:

- In XGBoost the decision is made with the second-order Gain formula.  
- In classical Gradient Boosting / CART the analogous complexity penalty is usually applied after the tree is grown (cost-complexity pruning) or via a simple minimum-impurity-decrease threshold that does not involve the Hessian.

Because XGBoost's Gain already incorporates both gradient and Hessian, γ interacts with the local curvature of the loss, giving a more statistically principled stopping rule.

### 04. How does XGBoost differ from standard Gradient Boosting?

| Aspect | Standard GBM | XGBoost |
|--------|--------------|---------|
| Loss approximation | First-order (gradient only) | Second-order (gradient + Hessian) |
| Leaf weight | $w = -\eta \frac{\sum g}{n}$ (or similar) | $w = -\frac{\sum g}{\sum h + \lambda}$ (Newton step) |
| Split criterion | Usually reduction in variance / MSE / Gini | Gain derived from the quadratic approximation of the loss |
| Regularisation | Learning rate, tree depth, min samples, post-pruning | Explicit λ (L2 on weights), γ (min gain), also column & row subsampling |
| Speed / engineering | Historically slower | Highly optimised (histogram, sparsity-aware, cache-aware, distributed) |

Using the Hessian turns the optimisation into a **Newton method** rather than pure gradient descent. This yields:
- better step sizes,
- natural instance weighting (points with larger second derivative influence the split more),
- a cleaner theoretical justification for the regularisation terms λ and γ.